# 56. The Safety Stock & Reorder Point (Q,r) Policy Problem

## Tier 3: Particle Swarm Optimization (Advanced Metaheuristic)

### Key Assumptions
- Particle Swarm Optimization can efficiently explore the (Q,r) solution space
- Each particle represents a potential solution with position (Q,r) and velocity
- Adaptive parameters prevent premature convergence while ensuring solution quality
- Swarm intelligence balances exploration of new solutions with exploitation of known good solutions

### Approach (Step-by-Step)
1. **Initialize swarm** with random particles representing (Q,r) solutions
2. **Define fitness function** incorporating holding, ordering, and stockout costs
3. **Update particle velocities** using PSO equations with adaptive inertia
4. **Update particle positions** and enforce solution bounds
5. **Track personal and global bests** across iterations
6. **Apply convergence criteria** to terminate optimization

### What to Look For in the Results
- Convergence behavior showing swarm learning over iterations
- Optimal (Q,r) parameters that minimize total cost
- Service level achievement compared to target requirements
- Performance comparison with heuristic and mathematical approaches

### Concrete Example: CardioStabil Medication
- Complex cost function with multiple local optima
- High service level requirement (99.5%) creates challenging optimization landscape
- Demand and lead time uncertainty adds non-linearity to fitness function
- PSO excels at finding global optima in such complex search spaces

In [1]:
# Import required libraries for Particle Swarm Optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for Particle Swarm Optimization")

Libraries imported successfully for Particle Swarm Optimization


Libraries imported successfully for Particle Swarm Optimization


Libraries imported successfully for Particle Swarm Optimization


Libraries imported successfully for Particle Swarm Optimization


Libraries imported successfully for Particle Swarm Optimization


In [ ]:
# Define the PSO Inventory Optimizer class
@dataclass
class DemandParameters:
    """Data class for demand parameters"""
    annual_demand: float
    daily_std: float
    lead_time: float
    lead_time_std: float

@dataclass
class CostParameters:
    """Data class for cost parameters"""
    unit_cost: float
    holding_rate: float
    ordering_cost: float
    stockout_cost: float

class PSOInventoryOptimizer:
    """Particle Swarm Optimization for (Q,r) inventory policy optimization"""
    
    def __init__(self, demand_params: DemandParameters, cost_params: CostParameters, 
                 service_level: float, swarm_size: int = 30, max_iterations: int = 100):
        """
        Initialize PSO optimizer with problem parameters.
        
        Args:
            demand_params: Demand characteristics
            cost_params: Cost structure parameters  
            service_level: Target service level (e.g., 0.995)
            swarm_size: Number of particles in the swarm
            max_iterations: Maximum number of iterations
        """
        self.demand_params = demand_params
        self.cost_params = cost_params
        self.service_level = service_level
        self.swarm_size = swarm_size
        self.max_iterations = max_iterations
        
        # Derive daily demand and z-score for service level
        self.daily_demand = demand_params.annual_demand / 365
        self.z_score = stats.norm.ppf(service_level)
        
        # PSO parameters: adaptive inertia, cognitive/social coefficients
        self.w_max, self.w_min = 0.9, 0.4  # Inertia weight bounds
        self.c1, self.c2 = 2.0, 2.0  # Cognitive and social coefficients
        
        # Define search bounds for Q (order quantity) and r (reorder point)
        self.q_bounds = [10, demand_params.annual_demand / 2]
        self.r_bounds = [
            self.daily_demand * demand_params.lead_time,
            self.daily_demand * demand_params.lead_time * 3
        ]
        
        # Initialize swarm of particles
        self.initialize_swarm()
        
    def initialize_swarm(self):
        """
        Create swarm with random positions (Q, r) and velocities.
        Each particle has position, velocity, personal best, and fitness.
        """
        self.particles = []
        
        for i in range(self.swarm_size):
            # Random position within bounds
            q = random.uniform(self.q_bounds[0], self.q_bounds[1])
            r = random.uniform(self.r_bounds[0], self.r_bounds[1])
            position = np.array([q, r])
            
            # Random velocity (smaller than search space)
            v_q = random.uniform(-0.1 * self.q_bounds[1], 0.1 * self.q_bounds[1])
            v_r = random.uniform(-0.1 * self.r_bounds[1], 0.1 * self.r_bounds[1])
            velocity = np.array([v_q, v_r])
            
            # Create particle dictionary
            particle = {
                'position': position,
                'velocity': velocity,
                'best_position': position.copy(),
                'best_fitness': float('inf'),
                'fitness': float('inf')
            }
            
            self.particles.append(particle)
        
        # Initialize global best (center of search space)
        self.global_best_position = np.array([
            self.q_bounds[1] / 2,
            self.r_bounds[1] / 2
        ])
        self.global_best_fitness = float('inf')
        
        print(f"Swarm initialized: {self.swarm_size} particles")
        print(f"Q bounds: [{self.q_bounds[0]:.1f}, {self.q_bounds[1]:.1f}]")
        print(f"r bounds: [{self.r_bounds[0]:.1f}, {self.r_bounds[1]:.1f}]")

print("PSO Inventory Optimizer class defined successfully")

In [ ]:
# Implement the fitness function and service level calculation
class PSOInventoryOptimizer(PSOInventoryOptimizer):
    
    def calculate_fitness(self, q: float, r: float) -> float:
        """
        Compute total annual cost (objective to minimize).
        Includes holding, ordering, stockout, and service penalty costs.
        
        Args:
            q: Order quantity
            r: Reorder point
            
        Returns:
            Total annual cost (lower is better)
        """
        # Ensure q, r > 0
        q, r = max(1, q), max(1, r)
        
        # Calculate expected lead time demand and safety stock
        expected_lead_demand = self.daily_demand * self.demand_params.lead_time
        safety_stock = max(0, r - expected_lead_demand)
        
        # Holding cost: (cycle stock + safety stock) × unit_cost × holding_rate
        cycle_stock = q / 2
        holding_cost = (cycle_stock + safety_stock) * self.cost_params.unit_cost * self.cost_params.holding_rate
        
        # Ordering cost: (annual_demand / q) × ordering_cost
        ordering_cost = (self.demand_params.annual_demand / q) * self.cost_params.ordering_cost
        
        # Service level calculation
        service_level_achieved = self._calculate_service_level(r)
        
        # Stockout cost: expected stockouts × stockout_cost
        stockout_prob = max(0, 1 - service_level_achieved)
        expected_stockouts = stockout_prob * self.demand_params.annual_demand / 4  # Approximation
        stockout_cost = expected_stockouts * self.cost_params.stockout_cost
        
        # Service penalty: punish not meeting target service level
        service_penalty = 0
        if service_level_achieved < self.service_level:
            service_penalty = (self.service_level - service_level_achieved) * 50000
        
        # Total cost
        total_cost = holding_cost + ordering_cost + stockout_cost + service_penalty
        
        return total_cost
    
    def _calculate_service_level(self, r: float) -> float:
        """
        Compute service level from reorder point r using normal distribution.
        
        Args:
            r: Reorder point
            
        Returns:
            Service level (0-1)
        """
        # Expected demand during lead time
        expected_demand = self.daily_demand * self.demand_params.lead_time
        
        # Combined standard deviation of demand during lead time
        demand_var = self.demand_params.lead_time * (self.demand_params.daily_std ** 2)
        lead_time_var = (self.daily_demand ** 2) * (self.demand_params.lead_time_std ** 2)
        combined_std = np.sqrt(demand_var + lead_time_var)
        
        # Handle edge case
        if combined_std == 0:
            return 1.0
        
        # Calculate z-score and service level
        z = (r - expected_demand) / combined_std
        service_level = stats.norm.cdf(z)
        
        return service_level

print("Fitness function and service level calculation implemented")

In [ ]:
# Implement particle update and PSO optimization loop
class PSOInventoryOptimizer(PSOInventoryOptimizer):
    
    def update_particle(self, particle: Dict, iteration: int):
        """
        Update particle velocity and position using PSO equations.
        
        Velocity update: v = w*v + c1*r1*(pbest - x) + c2*r2*(gbest - x)
        Position update: x = x + v
        """
        # Adaptive inertia weight (decreases linearly)
        w = self.w_max - (self.w_max - self.w_min) * iteration / self.max_iterations
        
        # Random coefficients for cognitive and social components
        r1, r2 = random.random(), random.random()
        
        # Cognitive component (personal best influence)
        cognitive = self.c1 * r1 * (particle['best_position'] - particle['position'])
        
        # Social component (global best influence)
        social = self.c2 * r2 * (self.global_best_position - particle['position'])
        
        # Update velocity
        particle['velocity'] = w * particle['velocity'] + cognitive + social
        
        # Update position
        particle['position'] += particle['velocity']
        
        # Enforce bounds
        particle['position'][0] = np.clip(particle['position'][0], self.q_bounds[0], self.q_bounds[1])
        particle['position'][1] = np.clip(particle['position'][1], self.r_bounds[0], self.r_bounds[1])
        
        # Evaluate fitness
        particle['fitness'] = self.calculate_fitness(particle['position'][0], particle['position'][1])
        
        # Update personal best if improved
        if particle['fitness'] < particle['best_fitness']:
            particle['best_fitness'] = particle['fitness']
            particle['best_position'] = particle['position'].copy()
        
        # Update global best if improved
        if particle['fitness'] < self.global_best_fitness:
            self.global_best_fitness = particle['fitness']
            self.global_best_position = particle['position'].copy()
    
    def optimize(self) -> Dict:
        """
        Main PSO optimization loop.
        
        Returns:
            Dictionary with optimization results
        """
        fitness_history = []
        
        # Initial evaluation of all particles
        print("Initial evaluation of swarm...")
        for particle in self.particles:
            particle['fitness'] = self.calculate_fitness(particle['position'][0], particle['position'][1])
            
            # Update personal best
            if particle['fitness'] < particle['best_fitness']:
                particle['best_fitness'] = particle['fitness']
                particle['best_position'] = particle['position'].copy()
            
            # Update global best
            if particle['fitness'] < self.global_best_fitness:
                self.global_best_fitness = particle['fitness']
                self.global_best_position = particle['position'].copy()
        
        fitness_history.append(self.global_best_fitness)
        
        # Main iteration loop
        print("Starting PSO optimization...")
        
        for iteration in range(self.max_iterations):
            # Update all particles
            for particle in self.particles:
                self.update_particle(particle, iteration)
            
            # Record best fitness
            fitness_history.append(self.global_best_fitness)
            
            # Early stopping check (minimal improvement in last 10 iterations)
            if iteration > 10:
                recent_improvement = (fitness_history[-10] - fitness_history[-1]) / fitness_history[-10]
                if recent_improvement < 0.001:
                    print(f"Early stopping at iteration {iteration + 1}")
                    break
            
            # Progress update
            if (iteration + 1) % 20 == 0:
                print(f"Iteration {iteration + 1}: Best Cost = ${self.global_best_fitness:,.2f}")
        
        # Calculate final results
        optimal_q, optimal_r = self.global_best_position[0], self.global_best_position[1]
        safety_stock = max(0, optimal_r - self.daily_demand * self.demand_params.lead_time)
        service_level_achieved = self._calculate_service_level(optimal_r)
        
        return {
            'Q': optimal_q,
            'r': optimal_r,
            'safety_stock': safety_stock,
            'total_cost': self.global_best_fitness,
            'service_level_achieved': service_level_achieved,
            'fitness_history': fitness_history,
            'iterations': iteration + 1,
            'converged': iteration < self.max_iterations - 1
        }

print("Particle update and PSO optimization loop implemented")

In [ ]:
# Initialize CardioStabil problem and run PSO optimization
# Define demand parameters
demand_params = DemandParameters(
    annual_demand=12000,
    daily_std=10,
    lead_time=14,
    lead_time_std=3
)

# Define cost parameters
cost_params = CostParameters(
    unit_cost=450,
    holding_rate=0.25,
    ordering_cost=200,
    stockout_cost=2000
)

# Target service level
service_level = 0.995

# Create and run PSO optimizer
print("=== INITIALIZING PSO OPTIMIZER ===")
optimizer = PSOInventoryOptimizer(
    demand_params=demand_params,
    cost_params=cost_params,
    service_level=service_level,
    swarm_size=40,
    max_iterations=150
)

print(f"\nTarget Service Level: {service_level*100:.1f}%")
print(f"Daily Demand: {optimizer.daily_demand:.1f} units")
print(f"Z-Score: {optimizer.z_score:.3f}")

# Run optimization
print("\n=== RUNNING PSO OPTIMIZATION ===")
result = optimizer.optimize()

# Display results
print("\n=== PSO OPTIMIZATION RESULTS ===")
print(f"Converged: {result['converged']}")
print(f"Iterations: {result['iterations']}")
print(f"Optimal Q: {result['Q']:.0f} units")
print(f"Optimal r: {result['r']:.0f} units")
print(f"Safety Stock: {result['safety_stock']:.0f} units")
print(f"Total Annual Cost: ${result['total_cost']:,.2f}")
print(f"Service Level Achieved: {result['service_level_achieved']:.4f} ({result['service_level_achieved']*100:.2f}%)")
print(f"Service Level Target: {service_level:.4f} ({service_level*100:.2f}%)")
print(f"Service Level Gap: {(result['service_level_achieved'] - service_level)*100:+.3f}%")

In [ ]:
# Create comprehensive PSO visualization
def create_pso_visualization(result, optimizer):
    """
    Create a 4-panel visualization showing:
    1. Convergence history
    2. Swarm evolution (particle positions)
    3. Service level achievement
    4. Cost component breakdown
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Particle Swarm Optimization - CardioStabil Inventory Policy', fontsize=16, fontweight='bold')
    
    # Panel 1: Convergence History
    fitness_history = result['fitness_history']
    iterations = range(len(fitness_history))
    
    ax1.plot(iterations, fitness_history, 'b-', linewidth=2, label='Best Fitness')
    ax1.fill_between(iterations, fitness_history, alpha=0.3, color='blue')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Total Annual Cost ($)')
    ax1.set_title(f'Convergence History (Converged: {result["converged"]})')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Mark final convergence point
    if result['converged']:
        ax1.axvline(x=result['iterations'], color='red', linestyle='--', 
                   label=f'Early Stop at {result["iterations"]}')
        ax1.legend()
    
    # Panel 2: Swarm Evolution (Final positions)
    # Extract final particle positions
    q_positions = [p['position'][0] for p in optimizer.particles]
    r_positions = [p['position'][1] for p in optimizer.particles]
    fitnesses = [p['fitness'] for p in optimizer.particles]
    
    # Normalize fitness for color mapping
    norm_fitnesses = (np.array(fitnesses) - min(fitnesses)) / (max(fitnesses) - min(fitnesses))
    
    scatter = ax2.scatter(q_positions, r_positions, c=norm_fitnesses, 
                          cmap='viridis', alpha=0.7, s=60, edgecolors='black')
    
    # Mark global best
    ax2.scatter(result['Q'], result['r'], color='red', s=200, marker='*', 
               edgecolors='black', linewidth=2, label='Global Best', zorder=5)
    
    ax2.set_xlabel('Order Quantity (Q)')
    ax2.set_ylabel('Reorder Point (r)')
    ax2.set_title('Final Swarm Positions')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add colorbar for fitness
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('Normalized Fitness')
    
    # Panel 3: Service Level Achievement
    target_sl = optimizer.service_level * 100
    achieved_sl = result['service_level_achieved'] * 100
    
    categories = ['Target', 'Achieved', 'Gap']
    values = [target_sl, achieved_sl, achieved_sl - target_sl]
    colors = ['lightblue', 'lightgreen', 'lightcoral' if achieved_sl >= target_sl else 'red']
    
    bars = ax3.bar(categories, values, color=colors, edgecolor='black', linewidth=1)
    ax3.set_ylabel('Service Level (%)')
    ax3.set_title('Service Level Achievement')
    ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + (0.5 if height >= 0 else -1),
                f'{value:+.3f}%', ha='center', va='bottom' if height >= 0 else 'top', fontweight='bold')
    
    # Panel 4: Cost Component Breakdown
    # Calculate cost components for optimal solution
    q, r = result['Q'], result['r']
    safety_stock = result['safety_stock']
    
    # Recalculate cost components
    cycle_stock = q / 2
    holding_cost = (cycle_stock + safety_stock) * cost_params.unit_cost * cost_params.holding_rate
    ordering_cost = (demand_params.annual_demand / q) * cost_params.ordering_cost
    
    service_level_achieved = result['service_level_achieved']
    stockout_prob = max(0, 1 - service_level_achieved)
    expected_stockouts = stockout_prob * demand_params.annual_demand / 4
    stockout_cost = expected_stockouts * cost_params.stockout_cost
    
    cost_components = ['Holding', 'Ordering', 'Stockout']
    cost_values = [holding_cost, ordering_cost, stockout_cost]
    colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
    
    wedges, texts, autotexts = ax4.pie(cost_values, labels=cost_components, colors=colors, 
                                      autopct='%1.1f%%', startangle=90)
    ax4.set_title(f'Cost Breakdown\nTotal: ${result["total_cost"]:,.0f}')
    
    plt.tight_layout()
    plt.show()

# Create visualization
create_pso_visualization(result, optimizer)

In [ ]:
# Performance comparison with previous tiers
def compare_with_previous_tiers(pso_result, optimizer):
    """
    Compare PSO results with mathematical formulation and priority-based heuristic.
    """
    # Mathematical formulation results (from Tier 1)
    math_results = {
        'method': 'Mathematical Formulation',
        'Q': 207,
        'r': 735,
        'safety_stock': 273,
        'total_cost': 67440.25,  # Approximate from Tier 1
        'service_level': 0.995
    }
    
    # Priority-based heuristic results (from Tier 2)
    heuristic_results = {
        'method': 'Priority-Based Heuristic',
        'Q': 207,  # From cost-focused strategy
        'r': 689,  # From cost-focused strategy
        'safety_stock': 227,
        'total_cost': 67440.25,  # Approximate from Tier 2
        'service_level': 0.995
    }
    
    # PSO results
    pso_results = {
        'method': 'Particle Swarm Optimization',
        'Q': pso_result['Q'],
        'r': pso_result['r'],
        'safety_stock': pso_result['safety_stock'],
        'total_cost': pso_result['total_cost'],
        'service_level': pso_result['service_level_achieved']
    }
    
    # Create comparison dataframe
    comparison_data = [math_results, heuristic_results, pso_results]
    comparison_df = pd.DataFrame(comparison_data)
    
    print("=== MULTI-TIER PERFORMANCE COMPARISON ===")
    print(comparison_df.round(2).to_string(index=False))
    
    # Calculate improvements
    baseline_cost = math_results['total_cost']
    pso_improvement = ((baseline_cost - pso_results['total_cost']) / baseline_cost) * 100
    heuristic_improvement = ((baseline_cost - heuristic_results['total_cost']) / baseline_cost) * 100
    
    print(f"\n=== IMPROVEMENT ANALYSIS ===")
    print(f"PSO vs Mathematical: {pso_improvement:+.2f}% cost improvement")
    print(f"Heuristic vs Mathematical: {heuristic_improvement:+.2f}% cost improvement")
    print(f"PSO vs Heuristic: {((heuristic_results['total_cost'] - pso_results['total_cost']) / heuristic_results['total_cost']) * 100:+.2f}%")
    
    # Visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Multi-Tier Performance Comparison', fontsize=14, fontweight='bold')
    
    # Panel 1: Cost Comparison
    methods = comparison_df['method']
    costs = comparison_df['total_cost']
    
    bars = ax1.bar(methods, costs, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax1.set_ylabel('Total Annual Cost ($)')
    ax1.set_title('Cost Comparison by Method')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add cost labels
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # Panel 2: Order Quantity Comparison
    q_values = comparison_df['Q']
    
    bars2 = ax2.bar(methods, q_values, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax2.set_ylabel('Order Quantity (Q)')
    ax2.set_title('Order Quantity Comparison')
    ax2.tick_params(axis='x', rotation=45)
    
    # Panel 3: Reorder Point Comparison
    r_values = comparison_df['r']
    
    bars3 = ax3.bar(methods, r_values, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax3.set_ylabel('Reorder Point (r)')
    ax3.set_title('Reorder Point Comparison')
    ax3.tick_params(axis='x', rotation=45)
    
    # Panel 4: Safety Stock Comparison
    ss_values = comparison_df['safety_stock']
    
    bars4 = ax4.bar(methods, ss_values, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax4.set_ylabel('Safety Stock (units)')
    ax4.set_title('Safety Stock Comparison')
    ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return comparison_df

# Run comparison
comparison_df = compare_with_previous_tiers(result, optimizer)

In [ ]:
# Parameter sensitivity analysis
def parameter_sensitivity_analysis():
    """
    Analyze PSO performance with different parameter settings.
    """
    # Test different swarm sizes
    swarm_sizes = [20, 40, 60]
    results = []
    
    for swarm_size in swarm_sizes:
        print(f"\nTesting swarm size: {swarm_size}")
        
        # Create optimizer with current swarm size
        test_optimizer = PSOInventoryOptimizer(
            demand_params=demand_params,
            cost_params=cost_params,
            service_level=service_level,
            swarm_size=swarm_size,
            max_iterations=100
        )
        
        # Run optimization
        test_result = test_optimizer.optimize()
        
        # Store results
        results.append({
            'Swarm_Size': swarm_size,
            'Total_Cost': test_result['total_cost'],
            'Iterations': test_result['iterations'],
            'Converged': test_result['converged'],
            'Service_Level': test_result['service_level_achieved'],
            'Q': test_result['Q'],
            'r': test_result['r']
        })
        
        print(f"  Cost: ${test_result['total_cost']:,.2f}")
        print(f"  Iterations: {test_result['iterations']}")
        print(f"  Converged: {test_result['converged']}")
    
    # Create results dataframe
    sensitivity_df = pd.DataFrame(results)
    
    print("\n=== SWARM SIZE SENSITIVITY ANALYSIS ===")
    print(sensitivity_df.round(2).to_string(index=False))
    
    # Visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('PSO Parameter Sensitivity Analysis', fontsize=14, fontweight='bold')
    
    # Panel 1: Cost vs Swarm Size
    ax1.plot(sensitivity_df['Swarm_Size'], sensitivity_df['Total_Cost'], 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Swarm Size')
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Cost vs Swarm Size')
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Iterations vs Swarm Size
    ax2.plot(sensitivity_df['Swarm_Size'], sensitivity_df['Iterations'], 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Swarm Size')
    ax2.set_ylabel('Iterations to Convergence')
    ax2.set_title('Convergence Speed vs Swarm Size')
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Service Level vs Swarm Size
    ax3.plot(sensitivity_df['Swarm_Size'], sensitivity_df['Service_Level']*100, 'go-', linewidth=2, markersize=8)
    ax3.axhline(y=service_level*100, color='red', linestyle='--', label=f'Target ({service_level*100:.1f}%)')
    ax3.set_xlabel('Swarm Size')
    ax3.set_ylabel('Service Level (%)')
    ax3.set_title('Service Level vs Swarm Size')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Solution Quality Distribution
    # Show Q and r values for different swarm sizes
    for i, row in sensitivity_df.iterrows():
        ax4.scatter(row['Q'], row['r'], s=100, alpha=0.7, 
                   label=f'Swarm {row["Swarm_Size"]}', edgecolors='black')
        ax4.annotate(f"{row['Swarm_Size']}", (row['Q'], row['r']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    ax4.set_xlabel('Order Quantity (Q)')
    ax4.set_ylabel('Reorder Point (r)')
    ax4.set_title('Solution Distribution by Swarm Size')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return sensitivity_df

# Run sensitivity analysis
sensitivity_results = parameter_sensitivity_analysis()

In [ ]:
# Summary and key insights
print("=== PARTICLE SWARM OPTIMIZATION SUMMARY ===")
print("\n🎯 ALGORITHM CHARACTERISTICS:")
print(f"   • Swarm Size: {optimizer.swarm_size} particles")
print(f"   • Max Iterations: {optimizer.max_iterations}")
print(f"   • Converged: {result['converged']}")
print(f"   • Actual Iterations: {result['iterations']}")
print(f"   • Inertia Weight: {optimizer.w_max:.1f} → {optimizer.w_min:.1f} (adaptive)")
print(f"   • Cognitive Coefficient: {optimizer.c1:.1f}")
print(f"   • Social Coefficient: {optimizer.c2:.1f}")

print("\n📊 OPTIMAL SOLUTION:")
print(f"   • Order Quantity (Q): {result['Q']:.0f} units")
print(f"   • Reorder Point (r): {result['r']:.0f} units")
print(f"   • Safety Stock: {result['safety_stock']:.0f} units")
print(f"   • Total Annual Cost: ${result['total_cost']:,.2f}")
print(f"   • Service Level Achieved: {result['service_level_achieved']*100:.2f}%")
print(f"   • Service Level Target: {service_level*100:.2f}%")

print("\n⚡ PSO ADVANTAGES:")
print("   • Global optimization capability")
print("   • Handles non-linear fitness functions")
print("   • Adaptive parameter control")
print("   • Swarm intelligence and collective learning")
print("   • Good convergence properties")
print("   • Parallelizable computation")

print("\n🔄 CONVERGENCE ANALYSIS:")
if result['converged']:
    print(f"   • Early convergence at iteration {result['iterations']}")
    print(f"   • Efficiency: {(result['iterations']/optimizer.max_iterations)*100:.1f}% of max iterations used")
else:
    print(f"   • Completed full {optimizer.max_iterations} iterations")
    print(f"   • May benefit from more iterations or parameter tuning")

print("\n📈 PERFORMANCE COMPARISON:")
baseline_cost = 67440.25  # Mathematical formulation baseline
improvement = ((baseline_cost - result['total_cost']) / baseline_cost) * 100
print(f"   • Cost Improvement vs Mathematical: {improvement:+.2f}%")
print(f"   • Solution Quality: {'Superior' if improvement > 0 else 'Comparable'}")

print("\n✅ PARTICLE SWARM OPTIMIZATION COMPLETE")
print("PSO demonstrates effective global optimization for complex (Q,r) problems,")
print("finding high-quality solutions through swarm intelligence and adaptive learning.")